In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

print("All libraries imported successfully!")

In [ ]:
# ============================================================
# Settings
# ============================================================

SEED = 42
DATASET_FILE = "creditcard.csv"

print("Settings configured successfully!")

In [ ]:
# ============================================================
# Check Dataset Location
# ============================================================

print("Current working directory:")
print(os.getcwd())

print("\nFiles in current directory:")
print(os.listdir())

if os.path.exists(DATASET_FILE):
    print("\ncreditcard.csv found successfully!")
else:
    print("\nERROR: creditcard.csv not found.")
    print("Place creditcard.csv in the current folder.")

In [ ]:
# ============================================================
# Load Dataset
# ============================================================

df = pd.read_csv(DATASET_FILE)

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

In [ ]:
# ============================================================
# Dataset Information
# ============================================================

print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nClass Distribution:")
print(df["Class"].value_counts())

In [ ]:
# ============================================================
# Check Class Imbalance
# ============================================================

fraud_percent = df["Class"].mean() * 100

print(
    f"Fraud transactions are only "
    f"{fraud_percent:.3f}% of the dataset."
)

print("\nThis is a highly imbalanced classification problem.")

In [ ]:
# ============================================================
# EDA - Class Distribution
# ============================================================

plt.figure(figsize=(7, 5))

sns.countplot(
    x="Class",
    data=df
)

plt.yscale("log")

plt.title("Legitimate vs Fraud Transactions")
plt.xlabel("Class (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Count (Log Scale)")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# EDA - Transaction Amount
# ============================================================

plt.figure(figsize=(7, 5))

sns.boxplot(
    x="Class",
    y="Amount",
    data=df
)

plt.ylim(0, 300)

plt.title("Transaction Amount by Class")
plt.xlabel("Class")
plt.ylabel("Transaction Amount")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Save EDA Overview
# ============================================================

plt.figure(figsize=(12, 5))

# Class distribution
plt.subplot(1, 2, 1)

sns.countplot(
    x="Class",
    data=df
)

plt.yscale("log")

plt.title("Legitimate vs Fraud Transactions")
plt.xlabel("Class")
plt.ylabel("Count (Log Scale)")


# Transaction amount
plt.subplot(1, 2, 2)

sns.boxplot(
    x="Class",
    y="Amount",
    data=df
)

plt.ylim(0, 300)

plt.title("Transaction Amount by Class")
plt.xlabel("Class")
plt.ylabel("Transaction Amount")

plt.tight_layout()

plt.savefig(
    "eda_overview.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: eda_overview.png")

In [ ]:
# ============================================================
# Scale Time and Amount
# ============================================================

scaler = StandardScaler()

df["scaled_amount"] = scaler.fit_transform(
    df["Amount"].values.reshape(-1, 1)
)

df["scaled_time"] = scaler.fit_transform(
    df["Time"].values.reshape(-1, 1)
)

df.drop(
    ["Time", "Amount"],
    axis=1,
    inplace=True
)

print("Time and Amount scaled successfully!")

print("\nUpdated Dataset:")
display(df.head())

In [ ]:
# ============================================================
# Prepare Features and Target
# ============================================================

X = df.drop(
    "Class",
    axis=1
)

y = df["Class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# ============================================================
# Train-Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# ============================================================
# Define Machine Learning Models
# ============================================================

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        class_weight="balanced",
        random_state=SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=10,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
}

print("Models created successfully!")

print("\nModels:")
for model_name in models:
    print("-", model_name)

In [ ]:
# ============================================================
# Train and Evaluate Models
# ============================================================

model_results = {}

plt.figure(figsize=(8, 6))

for name, model in models.items():

    print("\n")
    print("=" * 60)
    print("MODEL:", name)
    print("=" * 60)

    # Train model
    print("\nTraining model...")

    model.fit(
        X_train,
        y_train
    )

    # Predictions
    predictions = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )[:, 1]

    # Classification Report
    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            predictions,
            target_names=[
                "Legitimate",
                "Fraud"
            ]
        )
    )

    # Confusion Matrix
    print("Confusion Matrix:")

    cm = confusion_matrix(
        y_test,
        predictions
    )

    print(cm)

    # ROC-AUC
    auc = roc_auc_score(
        y_test,
        probabilities
    )

    print(
        f"\nROC-AUC Score: {auc:.4f}"
    )

    model_results[name] = auc

    # ROC Curve
    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc:.3f})"
    )

In [ ]:
# ============================================================
# ROC Curve
# ============================================================

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title(
    "ROC Curves for Fraud Detection Models"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "model_evaluation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: model_evaluation.png")